# 轨迹、掩码与奖励：一次错误评分会教出什么

状态：verified（仅下列 CPU 教学计算实际执行）。所有记录为人工构造，不是真实业务轨迹，也没有训练语言模型。

先读[轨迹数据](../01-concepts/01-trajectory-data.md)与[奖励投机](../01-concepts/04-verifiers-and-reward-hacking.md)，完整函数在[learning.py](../05-code/learning.py)。本实验依次检查组泄露、计算 SFT/DPO 损失、比较两种奖励，再算组内优势。

In [1]:
DOMAIN="12-agent-learning"
from pathlib import Path
import sys
# Find this topic whether Jupyter was started from the repository or lab directory.
ancestors = [Path.cwd(), *Path.cwd().parents]
repo = next(p for p in ancestors if (p / "10-Knowledge").is_dir())
sys.path.insert(0, str(repo / "10-Knowledge" / DOMAIN / "05-code"))
from learning import validate_splits, masked_nll, dpo_loss, group_advantages, weak_reward, guarded_reward
import math

## 1. 同一个问题的改写不能同时出现在训练和测试
先故意构造泄露，确认检查器会拒绝，再修正划分。

In [2]:
rows=[{"task_group":"bug17","split":"train"},{"task_group":"bug17","split":"test"}]
try:
    validate_splits(rows)
except ValueError as error:
    print(error)
else:
    raise AssertionError("leakage was not detected")
rows[1]["split"]="train"
print("valid groups:",validate_splits(rows))

split leakage: ['bug17']
valid groups: 1


## 2. SFT 掩码控制哪些位置承担损失
第三和第四个位置是模型要生成的动作参数。改变上下文位置的概率不会改变这个掩码损失；真实训练里上下文仍影响目标预测。

In [3]:
p=[.8,.8,.5,.25,.9,.9];mask=[0,0,1,1,0,0]
loss=masked_nll(p,mask)
changed=masked_nll([.1,.1,.5,.25,.1,.1],mask)
assert abs(loss-changed)<1e-12
print("masked loss:",round(loss,6))
assert abs(loss-1.0397207708399179)<1e-9

masked loss: 1.039721


## 3. DPO 衡量的是相对参考策略的偏好
四个输入都是序列对数概率。策略等于参考时损失为 log(2)；提高优选、降低较差序列的概率后，损失下降。这不证明独立任务成功率上升。

In [4]:
initial=dpo_loss(-2,-3,-2,-3,beta=.1)
improved=dpo_loss(-1,-4,-2,-3,beta=.1)
assert improved<initial
print({"initial":initial,"preference_better":improved})

{'initial': 0.6931471805599453, 'preference_better': 0.5981388693815918}


## 4. 让评分器相信自报结果会发生什么
候选一只打印 PASS，候选二真的通过外部验收，候选三越权后通过。外部验收字段在这里由我们手工给定；真实系统必须由可信测试环境写入。

In [5]:
records=[
 {"id":"fake","answer":"PASS","policy_allowed":True,"external_tests_passed":False},
 {"id":"valid","answer":"测试完成","policy_allowed":True,"external_tests_passed":True},
 {"id":"unauthorized","answer":"PASS","policy_allowed":False,"external_tests_passed":True}]
weak=[weak_reward(r) for r in records]
strong=[guarded_reward(r) for r in records]
print({"weak_rewards":weak,"guarded_rewards":strong})
assert weak==[1.,0.,1.] and strong==[0.,1.,-1.]
assert records[max(range(3),key=lambda i:weak[i])]["id"]=="fake"
assert records[max(range(3),key=lambda i:strong[i])]["id"]=="valid"

{'weak_rewards': [1.0, 0.0, 1.0], 'guarded_rewards': [0.0, 1.0, -1.0]}


## 5. GRPO 式组内优势只表达相对好坏
下面使用总体标准差。所有奖励相同就没有区分信号；在这个教学函数中优势均为零。

In [6]:
advantages=group_advantages([0.,1.,1.,0.])
print("mixed group:",advantages)
print("equal group:",group_advantages([1.,1.,1.]))
assert all(abs(a-b)<1e-6 for a,b in zip(advantages,[-1,1,1,-1]))
assert group_advantages([1.,1.,1.])==[0.,0.,0.]

mixed group: [-0.9999999800000003, 0.9999999800000003, 0.9999999800000003, -0.9999999800000003]
equal group: [0.0, 0.0, 0.0]


## 如何解释输出

实验验证了计算和四类已构造的失败边界。它没有优化任何模型参数；DPO 损失变化、优势正负以及获胜候选变化都不应被表述为“Agent 性能提升”。继续做真实更新可进入 [Tiny Transformer](../../../20-Projects/tiny-transformer/README.md)，但训练集 loss 降低依然不能替代独立任务效果。

练习：把三个候选的外部验收都设为失败，解释为什么多次采样仍没有正例信号；再构造同一任务组的不同措辞，观察仅文本哈希为什么不够。

自检参考：把外部验收都设为失败后，合法候选的 guarded reward 是 0，非法候选仍是 -1；评分器不该选出可执行的成功方案。若一组只有等分的合法失败，组内优势全为 0，不会凭空创造正确行为。只对文本做哈希会把同一问题的两种措辞当成不同样本，所以要先按原始任务来源分组，再划分集合。
